# ♟️ Modular Chess Engine V1.8 — Высокоскоростное обучение на 10M позиций в VRAM (NVIDIA L4)

Этот блокнот обучает **ModularChessNet-V1.8** (~17.7M параметров, `14 SE-ResBlocks × 256` каналов + пространственная **AlphaZero Conv2d Policy Head**) на **10 000 000 уникальных высококачественных позиций Stockfish**.

---

### 🚀 Ключевые улучшения архитектуры и пайплайна V1.8:
1. **📦 `onnxscript` и все пакеты ставятся в первой ячейке (`PYTORCH_CUDA_ALLOC_CONF` выставляется до `import torch`)** + мгновенный smoke-тест ONNX экспорта на `batch=1` и `batch=4`.
2. **🗜️ 8× Bit-Packing датасета в VRAM и кэш на Google Drive (~1.18 GB вместо 9.6 GB)**:
   * Бинарные плоскости `0..14` упакованы побитово в `[10M, 15, 8] uint8` (`~1.12 GiB`), а пустой 16-й канал (резерв для совместимости 1:1 с `fillPlanes()` в `js/ai_neural.js`) добавляется нулями на лету в `bfloat16` на GPU.
   * Готовый бинарный кэш сохраняется на Google Drive — при повторном запуске Colab все 10M позиций загружаются в VRAM за ~20 секунд.
3. **💎 Чистый отбор PV1, отсутствие дублей и детерминированный Train/Val сплит**:
   * Фильтр через **DuckDB** (`depth >= 20`, `knodes >= 50`), ранжирование внутри каждой группы `fen` по **`depth DESC, knodes DESC, оценка с учётом очереди хода DESC`** — гарантированно отбирается только лучший главный вариант (**PV1**) без побочных Multi-PV линий.
   * Кросс-шардовая дедупликация по 64-битному `blake2b`-хешу, безопасное скачивание через `.part` с ретраями и таймаутом, без `fork` после инициализации CUDA/DuckDB.
   * **Детерминированное разбиение по хешу (`pos_hash % 20 == 0` -> Val 5%)**: полностью исключает утечку Train→Val даже при обрыве и перезапуске Colab.
4. **⚖️ Баланс 10 000 000 позиций со ставкой на миттельшпиль**:
   * **Дебют (Opening):** `1 500 000` (**15%**)
   * **Миттельшпиль (Middlegame):** `5 500 000` (**55%**) — главная ставка на сложную середину игры
   * **Эндшпиль (Endgame):** `3 000 000` (**30%**)
5. **🧠 Пространственная AlphaZero Policy Head + калиброванная регуляризация + векторизованный EMA**:
   * Полносвёрточная `Conv2d(256→256, 3×3) → Conv2d(256→73, 1×1)` сохраняет пространственную привязку `from_sq × 73`.
   * `AdamW(fused=True)` с 2 группами параметров (`weight_decay=1e-4` только для `ndim >= 2`, `0.0` для BatchNorm/bias), `DropPath(0.02)`, `label_smoothing=0.015` только на `train` (на `val` — чистый CrossEntropy без сглаживания), `VALUE_WEIGHT = 1.6`, векторизованный EMA (`torch._foreach_lerp_`).
6. **⚡ Максимальный throughput на L4 и умное сохранение на Drive**:
   * `torch.compile`, `channels_last`, пошаговый **Warmup (1000 шагов) + Cosine LR**, ноль CPU–GPU синхронизаций внутри train и val циклов.
   * На Google Drive сохраняются лёгкие веса каждой эпохи (`epoch_XX_weights.pt`), единый возобновляемый `latest_checkpoint_v1_8.pt`, `best_model_v1_8.pt`, самодостаточный `modular_chess_v1_8.onnx` и графики.

---

In [ ]:
# 1. Настройка аллокатора CUDA до импорта PyTorch + установка onnxscript и зависимостей (в первых ячейках!)
import os

# Задаём переменную окружения строго ДО import torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q --upgrade onnxscript onnx onnxruntime duckdb pyarrow python-chess tqdm matplotlib

import torch
import onnx
import onnxscript
import onnxruntime as ort
import duckdb
import pyarrow

print("✅ Библиотеки для обучения и ONNX-экспорта установлены:")
print(f"   • PyTorch:     {torch.__version__}")
print(f"   • onnxscript:  {getattr(onnxscript, '__version__', 'installed')}")
print(f"   • onnx:        {onnx.__version__}")
print(f"   • onnxruntime: {ort.__version__}")
print(f"   • duckdb:      {duckdb.__version__}")
print(f"   • pyarrow:     {pyarrow.__version__}")

In [ ]:
# 2. Проверка GPU NVIDIA L4 и оптимизация Tensor Cores (BF16 / TF32)
if not torch.cuda.is_available():
    raise SystemError("⚠️ GPU не обнаружен! Выберите: Среда выполнения -> Сменить среду выполнения -> L4 GPU.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
capability = torch.cuda.get_device_capability(0)
use_bf16 = torch.cuda.is_bf16_supported()

torch.backends.cudnn.benchmark = True
if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=" * 66)
print(f"✅ Активный GPU:        {gpu_name}")
print(f"💾 Видеопамять (VRAM):  {total_vram_gb:.2f} GB")
print(f"⚡ Compute Capability:  {capability[0]}.{capability[1]}")
print(f"🚀 Режим AMP:           {'BF16 Native (Tensor Cores)' if use_bf16 else 'FP16 AMP'}")
print(f"🧵 Ядер CPU (Colab):    {os.cpu_count()}")
print("=" * 66)

In [ ]:
# 3. Подключение Google Drive и настройка сохранения чекпоинтов, best_model и кэша датасета
import shutil
from google.colab import drive

print("Подключение Google Drive...")
drive.mount("/content/drive")

LOCAL_WORK_DIR = "./v1_8_artifacts"
LOCAL_CKPT_DIR = os.path.join(LOCAL_WORK_DIR, "checkpoints")
DRIVE_DIR = "/content/drive/MyDrive/ModularChess_V1_8"
DRIVE_CKPT_DIR = os.path.join(DRIVE_DIR, "checkpoints")

os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

LOCAL_BEST_PT = os.path.join(LOCAL_WORK_DIR, "best_model_v1_8.pt")
LOCAL_BEST_ONNX = os.path.join(LOCAL_WORK_DIR, "modular_chess_v1_8.onnx")
DRIVE_BEST_PT = os.path.join(DRIVE_DIR, "best_model_v1_8.pt")
DRIVE_BEST_ONNX = os.path.join(DRIVE_DIR, "modular_chess_v1_8.onnx")
DRIVE_LATEST_CKPT = os.path.join(DRIVE_CKPT_DIR, "latest_checkpoint_v1_8.pt")
DRIVE_HISTORY_JSON = os.path.join(DRIVE_DIR, "training_history_v1_8.json")
DRIVE_DATASET_CACHE = os.path.join(DRIVE_DIR, "dataset_v1_8_10m_bitpacked.pt")


def save_and_sync_file(local_path: str, drive_path: str, label: str = ""):
    """Атомарное копирование файла на Google Drive с проверкой размера."""
    tmp_drive = drive_path + ".tmp"
    shutil.copyfile(local_path, tmp_drive)
    os.replace(tmp_drive, drive_path)
    loc_sz = os.path.getsize(local_path)
    drv_sz = os.path.getsize(drive_path)
    if loc_sz <= 0 or loc_sz != drv_sz:
        raise IOError(f"Ошибка проверки размера при сохранении {drive_path}: local={loc_sz}, drive={drv_sz}")
    tag = f"[{label}] " if label else ""
    print(f"   💾 {tag}Google Drive: {drive_path} ({drv_sz / (1024**2):.2f} MB)")


print(f"✅ Локальная папка:   {os.path.abspath(LOCAL_WORK_DIR)}")
print(f"✅ Папка Google Drive:{DRIVE_DIR}")
print(f"✅ Кэш датасета:      {DRIVE_DATASET_CACHE}")

In [ ]:
# 4. Архитектура ModularChessNet-V1.8 (14 SE-ResBlocks + Conv2d AlphaZero Policy Head) + smoke-тест ONNX
import copy
import inspect
import torch.nn as nn


class DropPath(nn.Module):
    """Stochastic Depth (активен только во время обучения, при eval/ONNX — identity)."""
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = x.new_empty(shape).bernoulli_(keep_prob)
        return x * random_tensor.div(keep_prob)


class SEBlock(nn.Module):
    def __init__(self, channels=256, reduction=4):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean((2, 3))
        y = self.relu(self.fc1(y))
        y = self.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y


class ResBlockSE(nn.Module):
    def __init__(self, channels=256, drop_path=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.se = SEBlock(channels, reduction=4)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = self.drop_path(out)
        return self.relu(out + res)


class ModularChessNetV18(nn.Module):
    """
    ModularChessNet V1.8:
    - Вход: [B, 16, 8, 8] (1:1 совместим с js/ai_neural.js fillPlanes())
    - Ствол: 14 SE-ResBlocks × 256 каналов + лёгкий DropPath(0.02)
    - Policy Head: полностью свёрточная голова AlphaZero
      Conv(256->256, 3x3) + BN + ReLU -> Conv(256->73, 1x1) -> permute(0,2,3,1).contiguous().flatten(1)
      Индекс (r*8+c)*73 + plane совпадает 1:1 с uci_to_4672() и moveTo4672()!
    - Value Head: Conv(256->4, 1x1) -> Linear(256->256) -> Linear(256->1) -> tanh [-1, +1]
    """
    def __init__(self, in_channels=16, num_res_blocks=14, channels=256, drop_path_rate=0.02):
        super().__init__()
        self.in_conv = nn.Conv2d(in_channels, channels, 3, padding=1, bias=False)
        self.in_bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

        dpr = [drop_path_rate * (i / max(1, num_res_blocks - 1)) for i in range(num_res_blocks)]
        self.blocks = nn.ModuleList([
            ResBlockSE(channels, drop_path=dpr[i]) for i in range(num_res_blocks)
        ])

        # Полносвёрточная AlphaZero Policy Head (без потери пространственной привязки from_sq)
        self.policy_conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.policy_bn1 = nn.BatchNorm2d(channels)
        self.policy_conv2 = nn.Conv2d(channels, 73, 1, bias=True)

        # Value Head (-1.0 .. +1.0 от перспективы белых)
        self.value_conv = nn.Conv2d(channels, 4, 1, bias=False)
        self.value_bn = nn.BatchNorm2d(4)
        self.value_fc1 = nn.Linear(4 * 8 * 8, 256)
        self.value_fc2 = nn.Linear(256, 1)

    def forward(self, x):
        out = self.relu(self.in_bn(self.in_conv(x)))
        for block in self.blocks:
            out = block(out)

        p = self.relu(self.policy_bn1(self.policy_conv1(out)))
        p = self.policy_conv2(p)  # [B, 73, 8, 8]
        policy_logits = p.permute(0, 2, 3, 1).contiguous().flatten(1)

        v = self.relu(self.value_bn(self.value_conv(out)))
        v = self.relu(self.value_fc1(v.flatten(1)))
        value = torch.tanh(self.value_fc2(v))

        return policy_logits, value


def unwrap_model(m: nn.Module) -> nn.Module:
    return getattr(m, "_orig_mod", m)


def export_selfcontained_onnx(src_model: nn.Module, onnx_path: str, verify: bool = True):
    """Экспортирует непрерывную NCHW-модель на CPU в единый .onnx файл и проверяет динамический batch."""
    base_model = unwrap_model(src_model)
    export_model = copy.deepcopy(base_model).cpu().to(memory_format=torch.contiguous_format).eval()
    dummy_in = torch.randn(1, 16, 8, 8, dtype=torch.float32)

    sig_params = inspect.signature(torch.onnx.export).parameters
    export_kwargs = dict(
        input_names=["board_state"],
        output_names=["policy_logits", "value"],
        dynamic_axes={
            "board_state": {0: "batch_size"},
            "policy_logits": {0: "batch_size"},
            "value": {0: "batch_size"},
        },
        opset_version=18,
    )
    if "external_data" in sig_params:
        export_kwargs["external_data"] = False

    with torch.no_grad():
        torch.onnx.export(export_model, dummy_in, onnx_path, **export_kwargs)

    data_sidecar = onnx_path + ".data"
    onnx_proto = onnx.load(onnx_path, load_external_data=True)
    onnx.save_model(onnx_proto, onnx_path, save_as_external_data=False)
    if os.path.exists(data_sidecar):
        os.remove(data_sidecar)

    if verify:
        onnx_checked = onnx.load(onnx_path)
        onnx.checker.check_model(onnx_checked)
        with open(onnx_path, "rb") as f:
            raw_bytes = f.read()
        sess = ort.InferenceSession(raw_bytes, providers=["CPUExecutionProvider"])
        for bs in (1, 4):
            test_np = torch.randn(bs, 16, 8, 8, dtype=torch.float32).numpy()
            out_p, out_v = sess.run(None, {"board_state": test_np})
            assert out_p.shape == (bs, 4672) and out_v.shape == (bs, 1)
        assert not os.path.exists(data_sidecar)

    del export_model


_smoke_model = ModularChessNetV18().to(device)
_n_params = sum(p.numel() for p in _smoke_model.parameters() if p.requires_grad)
_smoke_onnx = os.path.join(LOCAL_WORK_DIR, "_smoke_test_v1_8.onnx")
export_selfcontained_onnx(_smoke_model, _smoke_onnx, verify=True)
os.remove(_smoke_onnx)
del _smoke_model
torch.cuda.empty_cache()

print(f"🧠 ModularChessNet-V1.8 готова! Параметров: {_n_params:,} (~{_n_params/1e6:.2f}M)")
print("✅ Smoke-тест self-contained ONNX экспорта (batch=1 и batch=4) пройден!")

In [ ]:
# 5. Побитовое кодирование (15×8 байт на позицию), GPU-распаковка и классификатор фаз
import numpy as np

DIRECTIONS = [(-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1)]
KNIGHT_MOVES = [(-2, -1), (-2, 1), (-1, -2), (-1, 2), (1, -2), (1, 2), (2, -1), (2, 1)]
UNDERPROMOS = ["n", "b", "r"]

PIECE_MAP = {
    "P": 0, "N": 1, "B": 2, "R": 3, "Q": 4, "K": 5,
    "p": 6, "n": 7, "b": 8, "r": 9, "q": 10, "k": 11,
}
VAL_MAP = {"p": 1, "n": 3, "b": 3, "r": 5, "q": 9}

PHASE_OPENING = 0
PHASE_MIDDLEGAME = 1
PHASE_ENDGAME = 2

# Маски битов 0..7 для мгновенной распаковки [B, 15, 8] -> [B, 16, 8, 8] прямо на GPU
BIT_MASKS_GPU = torch.tensor([1, 2, 4, 8, 16, 32, 64, 128], dtype=torch.uint8, device=device).view(1, 1, 1, 8)


def uci_to_4672(uci: str):
    if not uci or len(uci) < 4:
        return None
    c0, c1, c2, c3 = uci[0], uci[1], uci[2], uci[3]
    if not ("a" <= c0 <= "h" and "1" <= c1 <= "8" and "a" <= c2 <= "h" and "1" <= c3 <= "8"):
        return None

    fc = ord(c0) - 97
    fr = 8 - (ord(c1) - 48)
    tc = ord(c2) - 97
    tr = 8 - (ord(c3) - 48)
    promo = uci[4].lower() if len(uci) > 4 else None

    from_sq = fr * 8 + fc
    dr = tr - fr
    dc = tc - fc

    if promo in UNDERPROMOS:
        p_idx = UNDERPROMOS.index(promo)
        p_dir = -1 if dc < 0 else (1 if dc > 0 else 0)
        return from_sq * 73 + 64 + (p_dir + 1) * 3 + p_idx

    for k_idx, (k_dr, k_dc) in enumerate(KNIGHT_MOVES):
        if dr == k_dr and dc == k_dc:
            return from_sq * 73 + 56 + k_idx

    for d_idx, (d_dr, d_dc) in enumerate(DIRECTIONS):
        for dist in range(1, 8):
            if dr == d_dr * dist and dc == d_dc * dist:
                return from_sq * 73 + d_idx * 7 + (dist - 1)
    return None


def pack_fen_into_row(fen: str, dst_row: np.ndarray):
    """
    Быстрая запись FEN напрямую в предвыделенный буфер (15, 8) uint8 без лишних аллокаций.
    Плоскость 15 остаётся 0 в js/ai_neural.js fillPlanes(), поэтому добавляется на GPU при распаковке.
    """
    dst_row.fill(0)
    parts = fen.split()
    board_str = parts[0]
    turn_str = parts[1] if len(parts) > 1 else "w"
    castling_str = parts[2] if len(parts) > 2 else "-"

    r, c = 0, 0
    for ch in board_str:
        if ch == "/":
            r += 1
            c = 0
        elif "1" <= ch <= "8":
            c += ord(ch) - 48
        else:
            p_idx = PIECE_MAP.get(ch)
            if p_idx is not None and r < 8 and c < 8:
                dst_row[p_idx, r] |= (1 << c)
            c += 1

    if turn_str == "w":
        dst_row[12, :] = 0xFF
    if "K" in castling_str:
        dst_row[13, 0:4] |= 0x0F
    if "Q" in castling_str:
        dst_row[13, 0:4] |= 0xF0
    if "k" in castling_str:
        dst_row[14, 4:8] |= 0x0F
    if "q" in castling_str:
        dst_row[14, 4:8] |= 0xF0


def unpack_boards_gpu(b_packed: torch.Tensor, out_dtype: torch.dtype, channels_last: bool = True) -> torch.Tensor:
    """Распаковка [B, 15, 8] uint8 -> [B, 16, 8, 8] на лету в видеопамяти GPU."""
    bs = b_packed.shape[0]
    unpacked_15 = ((b_packed.unsqueeze(-1) & BIT_MASKS_GPU) != 0).to(dtype=out_dtype)
    plane_15 = torch.zeros((bs, 1, 8, 8), dtype=out_dtype, device=b_packed.device)
    boards = torch.cat((unpacked_15, plane_15), dim=1)
    if channels_last:
        boards = boards.to(memory_format=torch.channels_last)
    return boards


def fen_to_tensor_reference(fen: str) -> np.ndarray:
    """Эталонная 16x8x8 функция (как в js/ai_neural.js) для самопроверки битовой распаковки и теста ONNX."""
    planes = np.zeros((16, 8, 8), dtype=np.float32)
    parts = fen.split()
    board_str = parts[0]
    turn_str = parts[1] if len(parts) > 1 else "w"
    castling_str = parts[2] if len(parts) > 2 else "-"
    r, c = 0, 0
    for ch in board_str:
        if ch == "/":
            r += 1
            c = 0
        elif "1" <= ch <= "8":
            c += ord(ch) - 48
        else:
            p_idx = PIECE_MAP.get(ch)
            if p_idx is not None and r < 8 and c < 8:
                planes[p_idx, r, c] = 1.0
            c += 1
    if turn_str == "w":
        planes[12, :, :] = 1.0
    if "K" in castling_str:
        planes[13, 0:4, 0:4] = 1.0
    if "Q" in castling_str:
        planes[13, 0:4, 4:8] = 1.0
    if "k" in castling_str:
        planes[14, 4:8, 0:4] = 1.0
    if "q" in castling_str:
        planes[14, 4:8, 4:8] = 1.0
    return planes


def classify_position_phase(fen: str) -> int:
    """
    Классификация фаз для 4- и 6-польных FEN:
    - ENDGAME:    фигур <= 14 (с королями), либо материал без королей <= 28, либо ход >= 32 при <= 18 фигурах
    - OPENING:    ход <= 10 (если есть) при >= 26 фигур, либо для 4-польных FEN: >= 28 фигур и >= 3 неразвитых на 1/8 горизонталях
    - MIDDLEGAME: всё остальное (основная ставка — 5.5M)
    """
    parts = fen.split()
    board_str = parts[0]
    castling_str = parts[2] if len(parts) > 2 else "-"
    total_pieces = 0
    mat_score = 0
    for ch in board_str:
        low = ch.lower()
        if low == "k":
            total_pieces += 1
        elif low in VAL_MAP:
            total_pieces += 1
            mat_score += VAL_MAP[low]

    if total_pieces <= 14 or mat_score <= 28:
        return PHASE_ENDGAME

    if len(parts) > 5 and parts[5].isdigit():
        fullmove = int(parts[5])
        if fullmove <= 10 and total_pieces >= 26:
            return PHASE_OPENING
        if fullmove >= 32 and total_pieces <= 18:
            return PHASE_ENDGAME
        return PHASE_MIDDLEGAME

    ranks = board_str.split("/")
    if total_pieces >= 28 and castling_str != "-" and len(ranks) == 8:
        back_rank_minors = sum(1 for ch in ranks[0] if ch in "nbq") + sum(1 for ch in ranks[7] if ch in "NBQ")
        if back_rank_minors >= 3:
            return PHASE_OPENING
    return PHASE_MIDDLEGAME


# Самопроверка bit-packing <-> GPU unpack и валидации UCI
_tmp_pk = np.zeros((1, 15, 8), dtype=np.uint8)
for _tf in (
    "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/3P1N2/PPP2PPP/RNBQ1RK1 b Kq e3",
    "8/5k2/8/8/8/8/1R6/4K3 w - -",
):
    pack_fen_into_row(_tf, _tmp_pk[0])
    _unp = unpack_boards_gpu(torch.from_numpy(_tmp_pk).to(device), torch.float32, channels_last=False)[0].cpu().numpy()
    assert np.array_equal(_unp, fen_to_tensor_reference(_tf)), f"Bitpack mismatch on {_tf}"

for _bad_uci in ("", "0000", "e9e4", "z1a1", "a0a2", None):
    assert uci_to_4672(_bad_uci) is None

print("✅ Bit-packing, GPU-распаковка, uci_to_4672 и 3-фазный классификатор проверены!")

In [ ]:
# 6. Загрузка 10M качественных PV1 позиций в bit-packed VRAM (без fork, с ретраями и кэшем Drive)
import hashlib
import json
import time
import urllib.request
from tqdm.auto import tqdm

# ==============================================================================
# КОНФИГУРАЦИЯ БАЛАНСА И КАЧЕСТВА ДАННЫХ V1.8 (10 000 000 ПОЗИЦИЙ)
# ==============================================================================
TARGET_TOTAL = 10_000_000
TARGET_OPENING = 1_500_000      # 15% — Дебют
TARGET_MIDDLEGAME = 5_500_000   # 55% — Главная ставка на Миттельшпиль!
TARGET_ENDGAME = 3_000_000      # 30% — Эндшпиль

MIN_DEPTH = 20                  # Отсекаем поверхностные расчёты (< 20 полуходов)
MIN_KNODES = 50                 # Минимум 50 000 узлов Stockfish
MAX_ABS_CP = 3000               # Клиппинг экстремальных cp-значений

USE_DRIVE_DATASET_CACHE = True  # Важно: при смене квот/фильтров удалите старый кэш с Google Drive
DELETE_SHARD_AFTER_INGEST = True
DATA_DIR = "./data_v1_8"
os.makedirs(DATA_DIR, exist_ok=True)


def discover_lichess_shards():
    api_url = "https://datasets-server.huggingface.co/parquet?dataset=Lichess%2Fchess-position-evaluations"
    try:
        req = urllib.request.Request(api_url, headers={"User-Agent": "ModularChess-V1.8"})
        with urllib.request.urlopen(req, timeout=20) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        files = [
            item["url"]
            for item in payload.get("parquet_files", [])
            if item.get("split") == "train" and item.get("url")
        ]
        if files:
            return files
    except Exception as e:
        print(f"ℹ️ HF Datasets-Server fallback ({e}), используем прямые ссылки refs/convert/parquet...")
    base = "https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/refs%2Fconvert/parquet/default/train"
    return [f"{base}/{i:04d}.parquet" for i in range(40)]


def download_shard_with_retries(url: str, dst_path: str, retries: int = 5, timeout_sec: int = 60):
    """Надёжное скачивание через .part с таймаутом, ретраями и проверкой Content-Length."""
    part_path = dst_path + ".part"
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "ModularChess-V1.8"})
            with urllib.request.urlopen(req, timeout=timeout_sec) as resp:
                expected_len = resp.headers.get("Content-Length")
                expected_bytes = int(expected_len) if expected_len and expected_len.isdigit() else None
                downloaded = 0
                with open(part_path, "wb") as out_f:
                    while True:
                        buf = resp.read(1024 * 1024)
                        if not buf:
                            break
                        out_f.write(buf)
                        downloaded += len(buf)
            if expected_bytes is not None and downloaded != expected_bytes:
                raise IOError(f"Неполное скачивание: {downloaded} != {expected_bytes}")
            os.replace(part_path, dst_path)
            return
        except Exception as err:
            if os.path.exists(part_path):
                os.remove(part_path)
            if attempt == retries:
                raise RuntimeError(f"Не удалось скачать {url} после {retries} попыток: {err}") from err
            wait_s = min(30, 2 ** attempt)
            print(f"   ⚠️ Сбой скачивания ({err}), повтор {attempt}/{retries} через {wait_s}с...")
            time.sleep(wait_s)


loaded_from_cache = False
if USE_DRIVE_DATASET_CACHE and os.path.exists(DRIVE_DATASET_CACHE):
    print(f"⚡ Найден bit-packed кэш датасета на Google Drive (~1.18 GB): {DRIVE_DATASET_CACHE}")
    cached = torch.load(DRIVE_DATASET_CACHE, map_location="cpu", weights_only=False)
    all_boards_gpu = cached["boards"].to(device, non_blocking=True)
    all_policies_gpu = cached["policies"].to(device, non_blocking=True)
    all_values_gpu = cached["values"].to(device, non_blocking=True)
    all_phases_gpu = cached["phases"].to(device, non_blocking=True)
    all_is_val_gpu = cached["is_val"].to(device, non_blocking=True)
    counts = list(cached["counts"])
    total_ingested = int(all_boards_gpu.shape[0])
    del cached
    torch.cuda.synchronize()
    loaded_from_cache = True
    print("✅ 10M датасет мгновенно загружен с Google Drive в VRAM!")

if not loaded_from_cache:
    shard_urls = discover_lichess_shards()
    print(f"📡 Доступно шардов: {len(shard_urls)} | Квоты: Дебют={TARGET_OPENING:,} (15%), Миттельшпиль={TARGET_MIDDLEGAME:,} (55%), Эндшпиль={TARGET_ENDGAME:,} (30%)")

    all_boards_gpu = torch.empty((TARGET_TOTAL, 15, 8), dtype=torch.uint8, device=device)
    all_policies_gpu = torch.empty(TARGET_TOTAL, dtype=torch.int16, device=device)
    all_values_gpu = torch.empty((TARGET_TOTAL, 1), dtype=torch.float16, device=device)
    all_phases_gpu = torch.empty(TARGET_TOTAL, dtype=torch.uint8, device=device)
    all_is_val_gpu = torch.empty(TARGET_TOTAL, dtype=torch.bool, device=device)

    counts = [0, 0, 0]
    quotas = [TARGET_OPENING, TARGET_MIDDLEGAME, TARGET_ENDGAME]
    seen_fen_hashes = set()
    written_total = 0

    CHUNK_SIZE = 32768
    chunk_boards = np.zeros((CHUNK_SIZE, 15, 8), dtype=np.uint8)
    chunk_policies = np.zeros(CHUNK_SIZE, dtype=np.int16)
    chunk_values = np.zeros((CHUNK_SIZE, 1), dtype=np.float16)
    chunk_phases = np.zeros(CHUNK_SIZE, dtype=np.uint8)
    chunk_isval = np.zeros(CHUNK_SIZE, dtype=np.bool_)
    chunk_count = 0

    def flush_chunk():
        global chunk_count, written_total
        if chunk_count == 0:
            return
        end_i = written_total + chunk_count
        all_boards_gpu[written_total:end_i].copy_(torch.from_numpy(chunk_boards[:chunk_count]), non_blocking=True)
        all_policies_gpu[written_total:end_i].copy_(torch.from_numpy(chunk_policies[:chunk_count]), non_blocking=True)
        all_values_gpu[written_total:end_i].copy_(torch.from_numpy(chunk_values[:chunk_count]), non_blocking=True)
        all_phases_gpu[written_total:end_i].copy_(torch.from_numpy(chunk_phases[:chunk_count]), non_blocking=True)
        all_is_val_gpu[written_total:end_i].copy_(torch.from_numpy(chunk_isval[:chunk_count]), non_blocking=True)
        written_total = end_i
        chunk_count = 0

    duck_tmp_dir = os.path.join(DATA_DIR, "duck_tmp")
    os.makedirs(duck_tmp_dir, exist_ok=True)
    duck_con = duckdb.connect(database=":memory:")
    duck_con.execute("SET memory_limit='8GB';")
    duck_con.execute("SET threads=4;")
    duck_con.execute(f"SET temp_directory='{duck_tmp_dir}';")

    pbar = tqdm(total=TARGET_TOTAL, desc="💎 Отбор 10M PV1 позиций")
    total_raw_scanned = 0
    total_dups_skipped = 0

    for s_idx, url in enumerate(shard_urls):
        if sum(counts) >= TARGET_TOTAL:
            break

        shard_path = os.path.join(DATA_DIR, f"shard_{s_idx:04d}.parquet")
        if not os.path.exists(shard_path):
            print(f"\n📥 Скачивание шарда {s_idx + 1}/{len(shard_urls)}: {os.path.basename(url)}...")
            download_shard_with_retries(url, shard_path)

        # В Multi-PV сортируем по depth DESC, knodes DESC И по лучшей оценке с точки зрения стороны на ходу -> гарантируем PV1!
        query = f"""
            WITH BestPerFen AS (
                SELECT
                    fen,
                    split_part(trim(line), ' ', 1) AS best_uci,
                    cp,
                    mate,
                    ROW_NUMBER() OVER (
                        PARTITION BY fen
                        ORDER BY
                            depth DESC,
                            knodes DESC,
                            (
                                CASE WHEN split_part(fen, ' ', 2) = 'w' THEN 1 ELSE -1 END
                                * CASE
                                    WHEN mate IS NOT NULL AND mate > 0 THEN 1000000 - CAST(mate AS BIGINT)
                                    WHEN mate IS NOT NULL AND mate < 0 THEN -1000000 - CAST(mate AS BIGINT)
                                    ELSE COALESCE(CAST(cp AS BIGINT), 0)
                                  END
                            ) DESC
                    ) AS rn
                FROM read_parquet('{shard_path}')
                WHERE depth >= {MIN_DEPTH}
                  AND knodes >= {MIN_KNODES}
                  AND line IS NOT NULL
                  AND length(trim(line)) >= 4
                  AND (cp IS NOT NULL OR (mate IS NOT NULL AND mate != 0))
            )
            SELECT fen, best_uci, cp, mate
            FROM BestPerFen
            WHERE rn = 1
        """

        rel = duck_con.sql(query)
        arrow_ctx = rel.to_arrow_reader(batch_size=65536) if hasattr(rel, "to_arrow_reader") else rel.fetch_record_batch(65536)

        for batch in arrow_ctx:
            rows = batch.to_pylist()
            total_raw_scanned += len(rows)

            for row in rows:
                fen = row["fen"]
                pol_idx = uci_to_4672(row["best_uci"])
                if pol_idx is None:
                    continue

                phase = classify_position_phase(fen)
                if counts[phase] >= quotas[phase]:
                    continue

                fen_core = " ".join(fen.split()[:4]).encode("ascii", errors="ignore")
                pos_hash = int.from_bytes(hashlib.blake2b(fen_core, digest_size=8).digest(), "little")
                if pos_hash in seen_fen_hashes:
                    total_dups_skipped += 1
                    continue

                mate = row["mate"]
                cp = row["cp"]
                if mate is not None and mate != 0:
                    val = 1.0 if mate > 0 else -1.0
                elif cp is not None:
                    c_clipped = max(-MAX_ABS_CP, min(MAX_ABS_CP, int(cp)))
                    val = float(np.tanh(c_clipped / 400.0))
                else:
                    continue

                seen_fen_hashes.add(pos_hash)
                counts[phase] += 1

                pack_fen_into_row(fen, chunk_boards[chunk_count])
                chunk_policies[chunk_count] = pol_idx
                chunk_values[chunk_count, 0] = val
                chunk_phases[chunk_count] = phase
                chunk_isval[chunk_count] = ((pos_hash % 20) == 0)
                chunk_count += 1
                pbar.update(1)

                if chunk_count >= CHUNK_SIZE:
                    flush_chunk()
                    pbar.set_postfix({"O": f"{counts[0]//1000}k", "M": f"{counts[1]//1000}k", "E": f"{counts[2]//1000}k"})

                if sum(counts) >= TARGET_TOTAL:
                    break

            if sum(counts) >= TARGET_TOTAL:
                break

        if hasattr(arrow_ctx, "close"):
            arrow_ctx.close()
        del arrow_ctx, rel

        if DELETE_SHARD_AFTER_INGEST and os.path.exists(shard_path):
            os.remove(shard_path)

    flush_chunk()
    torch.cuda.synchronize()
    pbar.set_postfix({"O": f"{counts[0]//1000}k", "M": f"{counts[1]//1000}k", "E": f"{counts[2]//1000}k"})
    pbar.close()
    duck_con.close()
    del seen_fen_hashes, chunk_boards, chunk_policies, chunk_values, chunk_phases, chunk_isval

    total_ingested = written_total
    if total_ingested < TARGET_TOTAL:
        all_boards_gpu = all_boards_gpu[:total_ingested]
        all_policies_gpu = all_policies_gpu[:total_ingested]
        all_values_gpu = all_values_gpu[:total_ingested]
        all_phases_gpu = all_phases_gpu[:total_ingested]
        all_is_val_gpu = all_is_val_gpu[:total_ingested]
        torch.cuda.empty_cache()

    print(f"\n🎉 Отобрано {total_ingested:,} уникальных позиций PV1 (проверено: {total_raw_scanned:,}, кросс-шард дублей отсеяно: {total_dups_skipped:,})")

    if USE_DRIVE_DATASET_CACHE and total_ingested > 0:
        local_ds_cache = os.path.join(LOCAL_WORK_DIR, "dataset_v1_8_10m_bitpacked.pt")
        print("💾 Сохранение bit-packed кэша (~1.18 GB) на Google Drive для мгновенных перезапусков...")
        torch.save({
            "boards": all_boards_gpu.cpu(),
            "policies": all_policies_gpu.cpu(),
            "values": all_values_gpu.cpu(),
            "phases": all_phases_gpu.cpu(),
            "is_val": all_is_val_gpu.cpu(),
            "counts": counts,
        }, local_ds_cache)
        save_and_sync_file(local_ds_cache, DRIVE_DATASET_CACHE, label="Dataset Cache")
        os.remove(local_ds_cache)

vram_used_gb = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"\n📊 Итоговый баланс в VRAM ({vram_used_gb:.2f} GB / {total_vram_gb:.2f} GB, свободно ~{total_vram_gb - vram_used_gb:.2f} GB):")
print(f"   • Дебют (Opening):       {counts[0]:,} ({counts[0]/max(1,total_ingested)*100:.2f}%)")
print(f"   • Миттельшпиль (Middle): {counts[1]:,} ({counts[1]/max(1,total_ingested)*100:.2f}%)")
print(f"   • Эндшпиль (Endgame):    {counts[2]:,} ({counts[2]/max(1,total_ingested)*100:.2f}%)")

In [ ]:
# 7. Обучение V1.8: 2 param groups AdamW, чистая Val CE, векторизованный EMA (_foreach_lerp_), torch.compile
import math
import time

# ==============================================================================
# ОПТИМИЗИРОВАННЫЕ ГИПЕРПАРАМЕТРЫ ОБУЧЕНИЯ V1.8
# ==============================================================================
BATCH_SIZE = 2048
EPOCHS = 12
WARMUP_STEPS = 1000              # Пошаговый линейный warmup (шаги 0..999)
PATIENCE = 4
BASE_LR = 1.5e-3
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4              # Применяется строго к весам (p.ndim >= 2), без BatchNorm/bias!
LABEL_SMOOTHING = 0.015          # Используется только на train; val — чистый CrossEntropy без сглаживания
VALUE_WEIGHT = 1.6
EMA_DECAY = 0.9995
GRAD_CLIP_NORM = 1.0
USE_TORCH_COMPILE = True
RESUME_FROM_CHECKPOINT = True

val_indices = torch.nonzero(all_is_val_gpu, as_tuple=False).squeeze(1).to(torch.int32)
train_indices = torch.nonzero(~all_is_val_gpu, as_tuple=False).squeeze(1).to(torch.int32)
train_size = int(train_indices.numel())
val_size = int(val_indices.numel())

print(f"📊 Детерминированный сплит (by hash): Train = {train_size:,} | Val = {val_size:,}")

amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

model = ModularChessNetV18(num_res_blocks=14, channels=256, drop_path_rate=0.02).to(
    device=device, memory_format=torch.channels_last
)
ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad_(False)

# Разделяем параметры: weight_decay только на 2D+ матрицы/свёртки, ноль на BatchNorm и bias
decay_params = [p for p in model.parameters() if p.requires_grad and p.ndim >= 2]
no_decay_params = [p for p in model.parameters() if p.requires_grad and p.ndim < 2]
param_groups = [
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": no_decay_params, "weight_decay": 0.0},
]
print(f"⚙️ AdamW param groups: decay={len(decay_params)} тензоров, no_decay={len(no_decay_params)} (BN/bias)")

try:
    optimizer = torch.optim.AdamW(param_groups, lr=BASE_LR, betas=(0.9, 0.98), fused=True)
    print("⚡ Оптимизатор: Fused AdamW")
except TypeError:
    optimizer = torch.optim.AdamW(param_groups, lr=BASE_LR, betas=(0.9, 0.98))

train_batches = train_size // BATCH_SIZE
val_batches = max(1, val_size // BATCH_SIZE)
total_steps = EPOCHS * train_batches


def step_lr_lambda(step_idx: int):
    if step_idx < WARMUP_STEPS:
        return 0.05 + 0.95 * (float(step_idx + 1) / float(max(1, WARMUP_STEPS)))
    progress = (step_idx - WARMUP_STEPS) / float(max(1, total_steps - WARMUP_STEPS))
    progress = min(1.0, max(0.0, progress))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_ratio = MIN_LR / BASE_LR
    return min_ratio + (1.0 - min_ratio) * cosine


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=step_lr_lambda)
train_policy_loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
val_policy_ce_none = nn.CrossEntropyLoss(reduction="none")  # Чистая валидационная CE без label smoothing!
value_mse_fn = nn.MSELoss()
value_huber_fn = nn.SmoothL1Loss(beta=0.25)
scaler = torch.amp.GradScaler("cuda", enabled=(not use_bf16))

start_epoch = 1
global_step = 0
best_val_loss = float("inf")
patience_counter = 0
history = []

if RESUME_FROM_CHECKPOINT and os.path.exists(DRIVE_LATEST_CKPT):
    print(f"🔄 Возобновление с последнего чекпоинта: {DRIVE_LATEST_CKPT}")
    ckpt = torch.load(DRIVE_LATEST_CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    if "ema_state_dict" in ckpt and ckpt["ema_state_dict"] is not None:
        ema_model.load_state_dict(ckpt["ema_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if ckpt.get("scaler_state_dict") and scaler.is_enabled():
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = int(ckpt["epoch"]) + 1
    global_step = int(ckpt.get("global_step", (start_epoch - 1) * train_batches))
    best_val_loss = float(ckpt.get("best_val_loss", float("inf")))
    patience_counter = int(ckpt.get("patience_counter", 0))
    history = list(ckpt.get("history", []))
    print(f"   ✅ Старт с эпохи {start_epoch} (step={global_step:,}, best_val={best_val_loss:.4f})")

# Предварительно собираем списки тензоров для сверхбыстрого векторизованного EMA (_foreach_lerp_)
ema_float_tensors, src_float_tensors = [], []
ema_int_tensors, src_int_tensors = [], []
for (_, et), (_, st) in zip(ema_model.state_dict().items(), model.state_dict().items()):
    if st.dtype.is_floating_point:
        ema_float_tensors.append(et)
        src_float_tensors.append(st)
    else:
        ema_int_tensors.append(et)
        src_int_tensors.append(st)

has_foreach_lerp = hasattr(torch, "_foreach_lerp_")

if USE_TORCH_COMPILE and hasattr(torch, "compile"):
    try:
        train_forward_model = torch.compile(model)
        print("⚡ torch.compile(model) активирован!")
    except Exception as ce:
        print(f"ℹ️ torch.compile пропущен ({ce}), работаем в eager/AMP режиме.")
        train_forward_model = model
else:
    train_forward_model = model


@torch.no_grad()
def update_ema(step_num: int):
    decay = min(EMA_DECAY, (step_num + 1.0) / (step_num + 10.0))
    one_minus = 1.0 - decay
    if has_foreach_lerp:
        torch._foreach_lerp_(ema_float_tensors, src_float_tensors, one_minus)
    else:
        torch._foreach_mul_(ema_float_tensors, decay)
        torch._foreach_add_(ema_float_tensors, src_float_tensors, alpha=one_minus)
    for et, st in zip(ema_int_tensors, src_int_tensors):
        et.copy_(st)


@torch.no_grad()
def evaluate_model(eval_net: nn.Module):
    eval_net.eval()
    v_loss = torch.zeros((), device=device, dtype=torch.float64)
    v_pol = torch.zeros((), device=device, dtype=torch.float64)
    v_mse = torch.zeros((), device=device, dtype=torch.float64)
    v_correct = torch.zeros((), device=device, dtype=torch.int64)
    v_total = 0

    phase_correct = [torch.zeros((), device=device, dtype=torch.int64) for _ in range(3)]
    phase_pol_sum = [torch.zeros((), device=device, dtype=torch.float64) for _ in range(3)]
    phase_total = [torch.zeros((), device=device, dtype=torch.int64) for _ in range(3)]

    for b_idx in range(val_batches):
        st = b_idx * BATCH_SIZE
        b_idx_t = val_indices[st : st + BATCH_SIZE].long()

        b_boards = unpack_boards_gpu(all_boards_gpu[b_idx_t], amp_dtype, channels_last=True)
        b_policies = all_policies_gpu[b_idx_t].long()
        b_values = all_values_gpu[b_idx_t].to(torch.float32)
        b_phases = all_phases_gpu[b_idx_t]

        with torch.amp.autocast("cuda", dtype=amp_dtype):
            pred_p, pred_v = eval_net(b_boards)
        per_ce = val_policy_ce_none(pred_p.float(), b_policies)
        loss_p = per_ce.mean()
        loss_v_mse = value_mse_fn(pred_v.float(), b_values)
        loss_v = 0.7 * loss_v_mse + 0.3 * value_huber_fn(pred_v.float(), b_values)
        loss = loss_p + VALUE_WEIGHT * loss_v

        v_loss += loss.detach()
        v_pol += loss_p.detach()
        v_mse += loss_v_mse.detach()

        matches = (pred_p.argmax(dim=1) == b_policies)
        v_correct += matches.sum()
        v_total += int(b_policies.numel())

        # Ноль синхронизаций на батче: булево маскирование и суммирование только на GPU
        for ph in (0, 1, 2):
            mask = (b_phases == ph)
            phase_total[ph] += mask.sum()
            phase_correct[ph] += (matches & mask).sum()
            phase_pol_sum[ph] += (per_ce * mask).sum()

    ph_tot_vals = [max(1, int(phase_total[ph].item())) for ph in range(3)]
    return {
        "loss": (v_loss / val_batches).item(),
        "policy": (v_pol / val_batches).item(),
        "mse": (v_mse / val_batches).item(),
        "top1_pct": (v_correct.item() / max(1, v_total)) * 100.0,
        "phase_top1": [phase_correct[ph].item() / ph_tot_vals[ph] * 100.0 for ph in range(3)],
        "phase_pol_loss": [phase_pol_sum[ph].item() / ph_tot_vals[ph] for ph in range(3)],
    }


print(f"\n🚀 Старт обучения V1.8: Эпохи {start_epoch}..{EPOCHS} ({train_batches:,} шагов/эпоха, warmup={WARMUP_STEPS} шагов)")
train_start_time = time.time()

for epoch in range(start_epoch, EPOCHS + 1):
    ep_start = time.time()
    model.train()
    t_perm = train_indices[torch.randperm(train_size, dtype=torch.int32, device=device)]

    t_loss_sum = torch.zeros((), device=device, dtype=torch.float64)
    t_pol_sum = torch.zeros((), device=device, dtype=torch.float64)
    t_val_mse_sum = torch.zeros((), device=device, dtype=torch.float64)

    ep_bar = tqdm(range(train_batches), desc=f"Эпоха {epoch:02d}/{EPOCHS} [Train]", leave=False)
    for b_idx in ep_bar:
        st = b_idx * BATCH_SIZE
        b_idx_t = t_perm[st : st + BATCH_SIZE].long()

        b_boards = unpack_boards_gpu(all_boards_gpu[b_idx_t], amp_dtype, channels_last=True)
        b_policies = all_policies_gpu[b_idx_t].long()
        b_values = all_values_gpu[b_idx_t].to(torch.float32)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=amp_dtype):
            pred_p, pred_v = train_forward_model(b_boards)
        loss_p = train_policy_loss_fn(pred_p.float(), b_policies)
        loss_v_mse = value_mse_fn(pred_v.float(), b_values)
        loss_v = 0.7 * loss_v_mse + 0.3 * value_huber_fn(pred_v.float(), b_values)
        loss = loss_p + VALUE_WEIGHT * loss_v

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

        scheduler.step()
        update_ema(global_step)
        global_step += 1

        t_loss_sum += loss.detach()
        t_pol_sum += loss_p.detach()
        t_val_mse_sum += loss_v_mse.detach()

        if (b_idx + 1) % 250 == 0 or (b_idx + 1) == train_batches:
            done_steps = float(b_idx + 1)
            ep_bar.set_postfix({
                "loss": f"{(t_loss_sum / done_steps).item():.4f}",
                "p": f"{(t_pol_sum / done_steps).item():.4f}",
                "v_mse": f"{(t_val_mse_sum / done_steps).item():.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.6f}",
            })

    del t_perm
    avg_train_loss = (t_loss_sum / train_batches).item()
    avg_train_pol = (t_pol_sum / train_batches).item()
    avg_train_mse = (t_val_mse_sum / train_batches).item()

    val_raw = evaluate_model(model)
    val_ema = evaluate_model(ema_model)
    use_ema_best = val_ema["loss"] <= val_raw["loss"]
    best_ep_val = val_ema if use_ema_best else val_raw
    winner_tag = "EMA" if use_ema_best else "RAW"
    winner_net = ema_model if use_ema_best else model

    curr_lr = scheduler.get_last_lr()[0]
    ep_mins = (time.time() - ep_start) / 60.0
    overfit_gap = best_ep_val["loss"] - avg_train_loss

    is_best = best_ep_val["loss"] < best_val_loss
    if is_best:
        impr = best_val_loss - best_ep_val["loss"]
        best_val_loss = best_ep_val["loss"]
        patience_counter = 0
    else:
        impr = 0.0
        patience_counter += 1

    epoch_rec = {
        "epoch": epoch,
        "global_step": global_step,
        "lr": curr_lr,
        "winner": winner_tag,
        "train_loss": avg_train_loss,
        "train_policy": avg_train_pol,
        "train_mse": avg_train_mse,
        "val_loss": best_ep_val["loss"],
        "val_raw_loss": val_raw["loss"],
        "val_ema_loss": val_ema["loss"],
        "val_policy": best_ep_val["policy"],
        "val_mse": best_ep_val["mse"],
        "val_top1_pct": best_ep_val["top1_pct"],
        "opening_top1_pct": best_ep_val["phase_top1"][0],
        "middlegame_top1_pct": best_ep_val["phase_top1"][1],
        "endgame_top1_pct": best_ep_val["phase_top1"][2],
        "opening_pol_loss": best_ep_val["phase_pol_loss"][0],
        "middlegame_pol_loss": best_ep_val["phase_pol_loss"][1],
        "endgame_pol_loss": best_ep_val["phase_pol_loss"][2],
        "overfit_gap": overfit_gap,
    }
    history.append(epoch_rec)

    print(f"\nЭпоха {epoch:02d}/{EPOCHS} ({ep_mins:.1f} мин) | LR: {curr_lr:.6f} | Best={winner_tag} | Gap(Val-Train): {overfit_gap:+.4f}")
    print(f"   • Train (smooth={LABEL_SMOOTHING}): Loss={avg_train_loss:.4f} | Policy={avg_train_pol:.4f} | Value MSE={avg_train_mse:.4f}")
    print(f"   • Val({winner_tag}, pure CE): Loss={best_ep_val['loss']:.4f} (raw={val_raw['loss']:.4f}, ema={val_ema['loss']:.4f}) | Policy={best_ep_val['policy']:.4f} | MSE={best_ep_val['mse']:.4f} | Top-1={best_ep_val['top1_pct']:.2f}%")
    print(
        f"   • По фазам: Дебют [ce={epoch_rec['opening_pol_loss']:.3f}, top1={epoch_rec['opening_top1_pct']:.2f}%] | "
        f"Миттельшпиль [ce={epoch_rec['middlegame_pol_loss']:.3f}, top1={epoch_rec['middlegame_top1_pct']:.2f}%] | "
        f"Эндшпиль [ce={epoch_rec['endgame_pol_loss']:.3f}, top1={epoch_rec['endgame_top1_pct']:.2f}%]"
    )

    # 1. На Drive: лёгкий архив весов каждой эпохи (~71 MB) + один возобновляемый resume-чекпоинт
    local_ep_weights = os.path.join(LOCAL_CKPT_DIR, f"epoch_{epoch:02d}_weights.pt")
    drive_ep_weights = os.path.join(DRIVE_CKPT_DIR, f"epoch_{epoch:02d}_weights.pt")
    torch.save(winner_net.state_dict(), local_ep_weights)
    save_and_sync_file(local_ep_weights, drive_ep_weights, label=f"Weights Ep{epoch:02d}")
    os.remove(local_ep_weights)

    local_latest = os.path.join(LOCAL_CKPT_DIR, "latest_checkpoint_v1_8.pt")
    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "arch": {"name": "ModularChessNetV18", "num_res_blocks": 14, "channels": 256, "drop_path_rate": 0.02},
        "model_state_dict": model.state_dict(),
        "ema_state_dict": ema_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler.is_enabled() else None,
        "best_val_loss": best_val_loss,
        "patience_counter": patience_counter,
        "history": history,
    }, local_latest)
    save_and_sync_file(local_latest, DRIVE_LATEST_CKPT, label="Resume Ckpt")

    with open(DRIVE_HISTORY_JSON, "w", encoding="utf-8") as hf:
        json.dump(history, hf, ensure_ascii=False, indent=2)

    # 2. Сохранение best_model + self-contained ONNX
    if is_best:
        torch.save(winner_net.state_dict(), LOCAL_BEST_PT)
        save_and_sync_file(LOCAL_BEST_PT, DRIVE_BEST_PT, label=f"BEST PT ({winner_tag})")
        try:
            export_selfcontained_onnx(winner_net, LOCAL_BEST_ONNX, verify=True)
            save_and_sync_file(LOCAL_BEST_ONNX, DRIVE_BEST_ONNX, label=f"BEST ONNX ({winner_tag})")
            print(f"   ⭐ НОВАЯ ЛУЧШАЯ МОДЕЛЬ! Val Loss улучшился на {impr:.4f} (best={best_val_loss:.4f})")
        except Exception as e:
            print(f"   ⚠️ Ошибка ONNX-экспорта: {e}")
    else:
        print(f"   ⏳ Val Loss не улучшился (Patience: {patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n🛑 Early Stopping сработал! Лучшая модель сохранена: {DRIVE_BEST_PT} и {DRIVE_BEST_ONNX}")
            break

total_mins = (time.time() - train_start_time) / 60.0
print("\n" + "=" * 66)
print(f"🎉 Обучение ModularChessNet-V1.8 завершено за {total_mins:.1f} мин!")
print(f"   • Лучший Val Loss: {best_val_loss:.4f}")
print(f"   • best_model PT:   {DRIVE_BEST_PT}")
print(f"   • best_model ONNX: {DRIVE_BEST_ONNX}")
print(f"   • Чекпоинты эпох:  {DRIVE_CKPT_DIR}")
print("=" * 66)

In [ ]:
# 8. Графики обучения (сохранение на Google Drive) и шахматная проверка modular_chess_v1_8.onnx
import chess
import matplotlib.pyplot as plt

if history:
    epochs_x = [r["epoch"] for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(epochs_x, [r["train_loss"] for r in history], label="Train Loss", marker="o")
    axes[0].plot(epochs_x, [r["val_loss"] for r in history], label="Val Best (Raw/EMA)", marker="s")
    axes[0].set_title("Total Loss (Train vs Val)")
    axes[0].set_xlabel("Эпоха")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(epochs_x, [r["train_mse"] for r in history], label="Train Value MSE", marker="o")
    axes[1].plot(epochs_x, [r["val_mse"] for r in history], label="Val Value MSE", marker="s")
    axes[1].set_title("Value Head MSE")
    axes[1].set_xlabel("Эпоха")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(epochs_x, [r["val_top1_pct"] for r in history], label="Val Top-1 Overall", marker="o", linewidth=2)
    axes[2].plot(epochs_x, [r["opening_top1_pct"] for r in history], label="Дебют", linestyle="--")
    axes[2].plot(epochs_x, [r["middlegame_top1_pct"] for r in history], label="Миттельшпиль", linestyle="--")
    axes[2].plot(epochs_x, [r["endgame_top1_pct"] for r in history], label="Эндшпиль", linestyle="--")
    axes[2].set_title("Policy Top-1 Accuracy (%)")
    axes[2].set_xlabel("Эпоха")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    curves_local = os.path.join(LOCAL_WORK_DIR, "training_curves_v1_8.png")
    curves_drive = os.path.join(DRIVE_DIR, "training_curves_v1_8.png")
    plt.savefig(curves_local, dpi=150)
    save_and_sync_file(curves_local, curves_drive, label="Training Curves PNG")
    plt.show()

onnx_to_test = LOCAL_BEST_ONNX if os.path.exists(LOCAL_BEST_ONNX) else DRIVE_BEST_ONNX
print(f"\n🔍 Тестирование итогового ONNX: {onnx_to_test}")
with open(onnx_to_test, "rb") as f:
    sess = ort.InferenceSession(f.read(), providers=["CPUExecutionProvider"])

test_fens = [
    ("Дебют (стартовая)", "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"),
    ("Миттельшпиль (тактика / жертва на f7)", "r1bqk2r/pppp1ppp/2n2n2/2b1p3/2B1P3/3P1N2/PPP2PPP/RNBQ1RK1 w kq - 2 6"),
    ("Миттельшпиль (мат в 1 Qxf7#)", "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5Q2/PPPP1PPP/RNB1K1NR w KQkq - 0 1"),
    ("Эндшпиль (ладейный)", "8/5k2/8/8/8/8/1R6/4K3 w - - 0 1"),
]

for label, fen in test_fens:
    board = chess.Board(fen)
    inp = fen_to_tensor_reference(fen)[np.newaxis, ...].astype(np.float32)
    logits, val = sess.run(None, {"board_state": inp})
    logits_1d = logits[0]
    val_white = float(val[0, 0])

    scored_moves = []
    for mv in board.legal_moves:
        idx = uci_to_4672(mv.uci())
        if idx is not None:
            scored_moves.append((float(logits_1d[idx]), mv.uci()))
    scored_moves.sort(reverse=True)
    top3 = ", ".join([f"{u} ({s:+.2f})" for s, u in scored_moves[:3]])
    print(f"♟️ {label}: WhiteEval={val_white:+.3f} | Top-3 хода: {top3}")

print("\n✅ Проверка завершена! Файл modular_chess_v1_8.onnx готов для подключения к движку.")